# Full Project Pipeline (Phase 1 -> Phase 8)

This notebook follows your project code path end-to-end with one cell per phase and try/except blocks for robust execution.

Current optimized setup:
- Phase 4 uses only 3 fast models: YOLO, ResNet50, EfficientNet-B0
- Phase 4 uses 3 fast algorithms: PSO, GWO, SA
- Phase 5 retrains dynamically based on Phase 4 output

In [ ]:
#for kaggle
# import os
# os.environ["RSNA_DATA_DIR"] = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"
# import shutil
# from pathlib import Path

# SRC = Path("/kaggle/input/datasets/moemenelsayed/mycode/Code")
# DST = Path("/kaggle/working/Code")

# # check source exists
# if not SRC.exists():
#     raise FileNotFoundError(f"Source folder not found: {SRC}")

# # remove old version safely
# if DST.exists():
#     shutil.rmtree(DST)

# # copy
# shutil.copytree(SRC, DST)

# print("✅ Copied to:", DST)

# # quick verification
# print("\nFiles inside Code:")
# for item in DST.iterdir():
#     print(" -", item)


In [ ]:
# 0) Install dependencies (Kaggle)
import importlib.util
import subprocess
import sys

required = {
    "ultralytics": "ultralytics==8.*",
    "albumentations": "albumentations",
    "pydicom": "pydicom",
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages already available.")

In [ ]:
# 0.1) Project path setup
import os
import sys
import json
from pathlib import Path

def _is_project_root(path: Path) -> bool:
    return path.is_dir() and (path / "src" / "config.py").is_file()

def find_project_root() -> Path:
    direct_candidates = [
        Path.cwd(),
        Path("/kaggle/working/Code"),  # main expected location after copy
        Path("/kaggle/input/datasets/moemenelsayed/mycode/Code"),  # fallback read-only source
    ]

    for candidate in direct_candidates:
        if _is_project_root(candidate):
            return candidate.resolve()

    for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if not base.exists():
            continue

        for child in base.iterdir():
            if _is_project_root(child):
                return child.resolve()

            if child.is_dir():
                for nested in child.iterdir():
                    if _is_project_root(nested):
                        return nested.resolve()

    raise FileNotFoundError(
        "Project root not found. Expected src/config.py inside /kaggle/working/Code "
        "or inside the attached Kaggle dataset."
    )

PROJECT_ROOT = find_project_root()

os.chdir(PROJECT_ROOT)

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

(PROJECT_ROOT / "artifacts").mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Working dir:", os.getcwd())
print("src/config.py exists:", (PROJECT_ROOT / "src" / "config.py").is_file())

In [ ]:
# 0.2) Imports from your project (same codebase)
from src.dataset import explore_dataset
from src.preprocessing import convert_dicom_to_png
from src.visualization import show_pneumonia_example
from src.yolo_dataset import build_yolo_dataset
from src.yolo_visualization import show_yolo_samples

from src.detection.train_yolo import train_yolo
from src.detection.train_fasterrcnn import train_fasterrcnn
from src.detection.train_retinanet import train_retinanet

from src.classification.train_resnet import train_resnet
from src.classification.train_densenet import train_densenet
from src.classification.train_efficientnet import train_efficientnet

from src.phase4_optimization import run_phase4_optimization
from src.phase5_retrain import run_phase5_retrain
from src.phase6_explainability import run_phase6_gradcam
from src.phase7_final_evaluation import run_phase7_final_evaluation
from src.phase8_demo import run_phase8_demo
from src.preflight import run_preflight_checks
from src.config import YOLO_DATASET_DIR

In [ ]:
# Config flags
RUN_PHASE1 = True
RUN_PHASE2 = True
RUN_PHASE3 = True
RUN_PHASE4 = True
RUN_PHASE5 = True
RUN_PHASE6 = True
RUN_PHASE7 = True
RUN_PHASE8 = True

QUICK_MODE = True
if QUICK_MODE:
    DET_BASELINE_EPOCHS = 1
    CLS_BASELINE_EPOCHS = 1
    OPT_QUICK_EPOCHS = 1
    OPT_POPULATION = 3
    OPT_ITERATIONS = 1
    DET_RETRAIN_EPOCHS = 2
    CLS_RETRAIN_EPOCHS = 2
else:
    DET_BASELINE_EPOCHS = 2
    CLS_BASELINE_EPOCHS = 3
    OPT_QUICK_EPOCHS = 1
    OPT_POPULATION = 4
    OPT_ITERATIONS = 2
    DET_RETRAIN_EPOCHS = 20
    CLS_RETRAIN_EPOCHS = 8

DEMO_IMAGE_SOURCE = os.path.join(YOLO_DATASET_DIR, "val", "images")
results = {}

## Phase 1 - Data Exploration and PNG Conversion

In [ ]:
df = None
preflight = run_preflight_checks()
results["preflight"] = preflight

if RUN_PHASE1:
    try:
        df = explore_dataset()
        convert_dicom_to_png()
        show_pneumonia_example(df)
        results["phase1"] = {"status": "ok"}
    except Exception as e:
        results["phase1"] = {"status": "failed", "error": str(e)}
        print("Phase 1 failed:", e)

## Phase 2 - Build YOLO Dataset and Visualization

In [ ]:
if RUN_PHASE2:
    try:
        if df is None:
            df = explore_dataset()
        build_yolo_dataset(df)
        show_yolo_samples()
        results["phase2"] = {"status": "ok"}
    except Exception as e:
        results["phase2"] = {"status": "failed", "error": str(e)}
        print("Phase 2 failed:", e)

## Phase 3 - Baseline Training (Per Model Try/Except)

In [ ]:
if RUN_PHASE3:
    phase3 = {}

    for name, fn in [
        ("yolo", lambda: train_yolo(epochs=DET_BASELINE_EPOCHS, run_name="phase3_yolo_baseline")),
        ("fasterrcnn", lambda: train_fasterrcnn(epochs=DET_BASELINE_EPOCHS)),
        ("retinanet", lambda: train_retinanet(epochs=DET_BASELINE_EPOCHS)),
    ]:
        try:
            phase3[name] = {"status": "ok", "metrics": fn()}
        except Exception as e:
            phase3[name] = {"status": "failed", "error": str(e)}
            print(f"Phase 3 {name} failed: {e}")

    for name, fn in [
        ("resnet50", lambda: train_resnet(epochs=CLS_BASELINE_EPOCHS)),
        ("densenet121", lambda: train_densenet(epochs=CLS_BASELINE_EPOCHS)),
        ("efficientnet_b0", lambda: train_efficientnet(epochs=CLS_BASELINE_EPOCHS)),
    ]:
        try:
            phase3[name] = {"status": "ok", "metrics": fn()}
        except Exception as e:
            phase3[name] = {"status": "failed", "error": str(e)}
            print(f"Phase 3 {name} failed: {e}")

    results["phase3"] = phase3
    with open("artifacts/phase3_baseline_results.json", "w", encoding="utf-8") as f:
        json.dump({k: v.get("metrics", {}) for k, v in phase3.items() if v.get("status") == "ok"}, f, indent=2)

## Phase 4 - Nature-Inspired Optimization (Fast Set: PSO, GWO, SA)

Optimized models in this fast run:
- YOLO
- ResNet50
- EfficientNet-B0

In [ ]:
if RUN_PHASE4:
    try:
        phase4 = run_phase4_optimization(
            quick_epochs=OPT_QUICK_EPOCHS,
            population=OPT_POPULATION,
            iterations=OPT_ITERATIONS,
        )
        results["phase4"] = {"status": "ok", "data": phase4}
    except Exception as e:
        results["phase4"] = {"status": "failed", "error": str(e)}
        print("Phase 4 failed:", e)

## Phase 5 - Retraining with Optimized Parameters

Uses dynamic retraining from `run_phase5_retrain()` based on whatever models were optimized in Phase 4.

In [ ]:
if RUN_PHASE5:
    try:
        phase5 = run_phase5_retrain(
            full_epochs_detection=DET_RETRAIN_EPOCHS,
            full_epochs_classification=CLS_RETRAIN_EPOCHS,
        )
        results["phase5"] = {"status": "ok", "data": phase5}
    except Exception as e:
        results["phase5"] = {"status": "failed", "error": str(e)}
        print("Phase 5 failed:", e)

## Phase 6 - Explainability (Grad-CAM)

In [ ]:
if RUN_PHASE6:
    try:
        phase6 = run_phase6_gradcam()
        results["phase6"] = {"status": "ok", "data": phase6}
    except Exception as e:
        results["phase6"] = {"status": "failed", "error": str(e)}
        print("Phase 6 failed:", e)

## Phase 7 - Final Evaluation

In [ ]:
if RUN_PHASE7:
    try:
        phase7 = run_phase7_final_evaluation()
        results["phase7"] = {"status": "ok", "data": phase7}
    except Exception as e:
        results["phase7"] = {"status": "failed", "error": str(e)}
        print("Phase 7 failed:", e)

## Phase 8 - Final Demo

In [ ]:
if RUN_PHASE8:
    try:
        if os.path.isdir(DEMO_IMAGE_SOURCE):
            files = sorted([f for f in os.listdir(DEMO_IMAGE_SOURCE) if f.endswith(".png")])
            if not files:
                raise RuntimeError("No PNG image found in demo folder.")
            demo_image = os.path.join(DEMO_IMAGE_SOURCE, files[0])
        else:
            demo_image = DEMO_IMAGE_SOURCE

        phase8 = run_phase8_demo(demo_image)
        results["phase8"] = {"status": "ok", "data": phase8}
        print("Demo output:", phase8)
    except Exception as e:
        results["phase8"] = {"status": "failed", "error": str(e)}
        print("Phase 8 failed:", e)

## Save Summary

In [ ]:
import os, json, glob, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path

ROOT = "/kaggle/working/Code"
ART = os.path.join(ROOT, "artifacts")
RUNS = os.path.join(ROOT, "runs")
HANDOFF = os.path.join(ART, "web_handoff")
os.makedirs(ART, exist_ok=True)
os.makedirs(HANDOFF, exist_ok=True)

def rel(p):
    return os.path.relpath(p, ROOT).replace("\\", "/")

def load_json_if_exists(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def latest_best_pt():
    cands = glob.glob(os.path.join(RUNS, "**", "weights", "best.pt"), recursive=True)
    if not cands:
        return None
    cands.sort(key=os.path.getmtime, reverse=True)
    return cands[0]

def safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

now_iso = datetime.now(timezone.utc).isoformat()
phase3_path = os.path.join(ART, "phase3_baseline_results.json")
phase4_path = os.path.join(ART, "phase4_best_hyperparameters.json")
phase5_path = os.path.join(ART, "phase5_retrain_results.json")
phase6_path = os.path.join(ART, "phase6_gradcam_results.json")
phase7_path = os.path.join(ART, "phase7_final_evaluation.json")
phase8_path = os.path.join(ART, "phase8_demo_result.json")
demo_img_path = os.path.join(ART, "demo_output.png")

phase3 = load_json_if_exists(phase3_path)
phase4 = load_json_if_exists(phase4_path)
phase5 = load_json_if_exists(phase5_path)
phase8 = load_json_if_exists(phase8_path)
best_pt = latest_best_pt()
gradcams = sorted(glob.glob(os.path.join(ART, "gradcam", "*.png")))

best_det = None
best_cls = None
if isinstance(phase4, dict) and phase4:
    items = {k: v for k, v in phase4.items() if isinstance(v, dict) and k != "_errors"}
    det_candidates = {k: v for k, v in items.items() if v.get("task") == "detection"}
    cls_candidates = {k: v for k, v in items.items() if v.get("task") == "classification"}
    if det_candidates:
        best_det = max(det_candidates, key=lambda k: det_candidates[k].get("best_score", -1.0))
    if cls_candidates:
        best_cls = max(cls_candidates, key=lambda k: cls_candidates[k].get("best_score", -1.0))

demo_detected = None
demo_conf = None
if isinstance(phase8, dict):
    demo_detected = phase8.get("detected")
    demo_conf = safe_float(phase8.get("confidence"))

have_phase3 = os.path.exists(phase3_path)
have_weights = best_pt is not None
status = "COMPLETED" if (have_phase3 and have_weights) else "FAILED"

payload = {
    "job_id": f"rsna_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}",
    "status": status,
    "created_at": now_iso,
    "finished_at": now_iso,
    "project_root": ROOT,
    "artifacts": {
        "phase3_baseline_results": rel(phase3_path) if os.path.exists(phase3_path) else None,
        "phase4_best_hyperparameters": rel(phase4_path) if os.path.exists(phase4_path) else None,
        "phase5_retrain_results": rel(phase5_path) if os.path.exists(phase5_path) else None,
        "phase6_gradcam_results": rel(phase6_path) if os.path.exists(phase6_path) else None,
        "phase7_final_evaluation": rel(phase7_path) if os.path.exists(phase7_path) else None,
        "phase8_demo_result": rel(phase8_path) if os.path.exists(phase8_path) else None,
        "demo_output_image": rel(demo_img_path) if os.path.exists(demo_img_path) else None,
        "gradcam_images": [rel(p) for p in gradcams],
        "best_yolo_weights": rel(best_pt) if best_pt else None
    },
    "summary": {
        "best_detection_model": best_det,
        "best_classification_model": best_cls,
        "demo_detected": demo_detected,
        "demo_confidence": demo_conf
    },
    "error": None if status == "COMPLETED" else "Missing baseline results or YOLO weights"
}

notebook_summary_path = os.path.join(ART, "kaggle_notebook_summary.json")
with open(notebook_summary_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

web_result_path = os.path.join(ART, "web_result.json")
with open(web_result_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

for old_file in glob.glob(os.path.join(HANDOFF, "*")):
    if os.path.isfile(old_file):
        os.remove(old_file)

if best_pt:
    shutil.copy2(best_pt, os.path.join(HANDOFF, "yolo_best.pt"))

copy_candidates = [
    phase3_path,
    phase4_path,
    phase5_path,
    phase6_path,
    phase7_path,
    phase8_path,
    demo_img_path,
    notebook_summary_path,
    web_result_path,
]
for src_path in copy_candidates:
    if os.path.exists(src_path):
        shutil.copy2(src_path, os.path.join(HANDOFF, os.path.basename(src_path)))

manifest = {
    "created_at_utc": now_iso,
    "model_type": "ultralytics_yolo_detection",
    "weights_file": "yolo_best.pt" if best_pt else None,
    "task": "pneumonia_detection",
    "class_names": ["pneumonia"],
    "default_imgsz": 640,
    "default_conf": 0.25,
    "status": status,
    "handoff_files": sorted(os.listdir(HANDOFF)),
}
manifest_path = os.path.join(HANDOFF, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

zip_path = os.path.join(ART, "web_handoff.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path(HANDOFF).glob("*")):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Saved:", notebook_summary_path)
print("Saved:", web_result_path)
print("Saved:", manifest_path)
print("Saved:", zip_path)
print("Handoff files:")
for p in sorted(Path(HANDOFF).glob("*")):
    if p.is_file():
        print("-", p.as_posix())
